# Week 3: Multi-Layer SEP with Entropy Analysis

## The Best of Both Worlds

**Combines:**
1. ✅ SEP's correct training objective (uncertainty TYPE not semantic mode)
2. ✅ Multi-layer probe (layers [8, 16, 31] capture distributed info)
3. ✅ Proper entropy computation (H_code and H_lang separately)
4. ✅ Diverse test set (validates real learning, not memorization)

## Approach

```
Prompt: "import"
    |
    v
Extract hidden states at layers [8, 16, 31]
    |
    v
Concatenate: h ∈ R^(3×4096) = R^12288
    |
    v
Multi-layer SEP probe
    |
    v
Predict: P(code_uncertainty | h)
    |
    v
Generate next token distribution
    |
    v
Classify tokens → compute H_code, H_lang
    |
    v
CCE = (P_code × H_code) - (P_lang × H_lang)
```

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy, ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()
vocab_size = len(tokenizer)

print(f"✅ Model loaded on {model.device}")
print(f"Vocabulary size: {vocab_size:,}")

## 1. Train Multi-Layer SEP Probe

In [ ]:
# Cell 4: Training data (SEP style - labels uncertainty TYPE)

TRAIN_EXAMPLES = [
    # CODE UNCERTAINTY (label = 1)
    {'prompt': 'import', 'label': 1, 'desc': 'Uncertain which module'},
    {'prompt': 'from sklearn import', 'label': 1, 'desc': 'Uncertain sklearn module'},
    {'prompt': 'def process_data(df):\n    df.', 'label': 1, 'desc': 'Uncertain pandas method'},
    {'prompt': 'const [state, setState] = use', 'label': 1, 'desc': 'Uncertain React hook'},
    {'prompt': 'async function fetch_data() {\n    await', 'label': 1, 'desc': 'Uncertain async op'},
    {'prompt': 'model = tf.keras.', 'label': 1, 'desc': 'Uncertain Keras class'},
    {'prompt': 'app = FastAPI()\n@app.', 'label': 1, 'desc': 'Uncertain FastAPI decorator'},
    {'prompt': 'SELECT * FROM users WHERE', 'label': 1, 'desc': 'Uncertain SQL condition'},
    {'prompt': 'git ', 'label': 1, 'desc': 'Uncertain git command'},
    {'prompt': 'docker run -', 'label': 1, 'desc': 'Uncertain docker flag'},

    # LANGUAGE UNCERTAINTY (label = 0)
    {'prompt': 'This function', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'The algorithm is', 'label': 0, 'desc': 'Uncertain which adjective'},
    {'prompt': 'Code quality can be', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'Explain what this code', 'label': 0, 'desc': 'Uncertain which verb'},
    {'prompt': 'The main advantage of async programming is', 'label': 0, 'desc': 'Uncertain benefit'},
    {'prompt': 'TypeScript provides better', 'label': 0, 'desc': 'Uncertain improvement'},
    {'prompt': 'Recursion is useful when', 'label': 0, 'desc': 'Uncertain scenario'},
    {'prompt': 'REST APIs are designed to', 'label': 0, 'desc': 'Uncertain purpose'},
    {'prompt': 'The difference between let and const is', 'label': 0, 'desc': 'Uncertain explanation'},
    {'prompt': 'Unit tests help', 'label': 0, 'desc': 'Uncertain benefit'},
]

print(f"Training examples: {len(TRAIN_EXAMPLES)}")
print(f"  Code uncertainty: {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 1)}")
print(f"  Language uncertainty: {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 0)}")

In [ ]:
# Cell 5: Extract multi-layer hidden states

SELECTED_LAYERS = [8, 16, 31]  # Best from Multi-Layer MLP Probe

def get_multi_layer_state(prompt: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
X_train = []
y_train = []

for example in tqdm(TRAIN_EXAMPLES, desc="Extracting"):
    h = get_multi_layer_state(example['prompt'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(example['label'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"✅ Hidden states: {X_train.shape}")
print(f"   Labels: {y_train.shape}")

In [ ]:
# Cell 6: Train multi-layer SEP probe with cross-validation

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Leave-One-Out Cross-Validation
probe = LogisticRegression(max_iter=1000, random_state=42)
loo = LeaveOneOut()
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=loo)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"✅ Multi-Layer SEP Probe (Cross-Validation)")
print(f"   Layers: {SELECTED_LAYERS}")
print(f"   Hidden dim: {X_train.shape[1]:,}")
print(f"   CV Accuracy: {cv_accuracy:.0%} ({int(cv_accuracy * len(y_train))}/{len(y_train)})")

# Check errors
errors = [(TRAIN_EXAMPLES[i], y_pred_cv[i]) for i in range(len(y_train)) if y_train[i] != y_pred_cv[i]]
if errors:
    print(f"\n   Errors:")
    for ex, pred in errors:
        true_str = 'CODE' if ex['label'] == 1 else 'LANG'
        pred_str = 'CODE' if pred == 1 else 'LANG'
        print(f"     '{ex['prompt'][:40]}...' -> predicted {pred_str}, actual {true_str}")
else:
    print("\n   No errors! Perfect CV accuracy.")

# Train final probe on all data
probe.fit(X_train_scaled, y_train)
print(f"\n✅ Final probe trained on all {len(y_train)} examples")

## 2. Entropy Computation Functions

In [ ]:
# Cell 7: Token classification

PURE_CODE_TOKENS = {
    'def', 'class', 'import', 'from', 'return', 'yield', 'async', 'await',
    'function', 'const', 'let', 'var', 'export', 'require', 'module',
    'SELECT', 'FROM', 'WHERE', 'INSERT', 'UPDATE', 'DELETE',
    '{', '}', '(', ')', '[', ']', ';', '=>',
}

PURE_LANGUAGE_TOKENS = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
    'what', 'how', 'why', 'when', 'where', 'which',
    'this', 'that', 'these', 'those',
}

STRUCTURAL_TOKENS = {
    '\n', '\r', '\t', '    ', '  ',
    ',', '.', ':', '+', '-', '*', '/', '=', '<', '>', '&', '|',
}

def classify_token_by_probe(token: str, uncertainty_type: int) -> str:
    """
    Classify token based on probe's uncertainty type prediction.
    
    uncertainty_type = 1 → code uncertainty (uncertain which code element)
    uncertainty_type = 0 → language uncertainty (uncertain which word)
    """
    token_clean = token.strip().lower()
    
    # Structural tokens are always 'other'
    if token in STRUCTURAL_TOKENS or token_clean in STRUCTURAL_TOKENS:
        return 'other'
    
    # Pure code/language tokens
    if token_clean in PURE_CODE_TOKENS:
        return 'code'
    if token_clean in PURE_LANGUAGE_TOKENS:
        return 'language'
    
    # AMBIGUOUS: Use probe!
    if uncertainty_type == 1:  # Code uncertainty
        return 'code'
    else:  # Language uncertainty
        return 'language'

print("✅ Token classification function ready")

In [ ]:
# Cell 8: Entropy computation

def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    """Compute entropy in bits."""
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def compute_entropy_breakdown(prompt: str, probe, scaler, layers: List[int]) -> Dict:
    """
    For a given prompt:
    1. Extract multi-layer hidden state
    2. Probe predicts uncertainty type
    3. Get next token logits
    4. Classify all tokens based on uncertainty type
    5. Compute H_code, H_lang, and CCE
    """
    # Step 1: Extract multi-layer hidden state
    h = get_multi_layer_state(prompt, layers).reshape(1, -1)
    h_scaled = scaler.transform(h)
    
    # Step 2: Probe predicts uncertainty type
    uncertainty_type = probe.predict(h_scaled)[0]
    uncertainty_prob = probe.predict_proba(h_scaled)[0, 1]  # P(code uncertainty)
    
    # Step 3: Get next token logits
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    
    # Step 4: Classify all tokens
    code_indices = []
    lang_indices = []
    other_indices = []
    
    for token_id in range(len(logits)):
        token_str = tokenizer.decode([token_id])
        token_class = classify_token_by_probe(token_str, uncertainty_type)
        
        if token_class == 'code':
            code_indices.append(token_id)
        elif token_class == 'language':
            lang_indices.append(token_id)
        else:
            other_indices.append(token_id)
    
    # Step 5: Compute entropy and CCE
    P_code = np.sum(probs[code_indices]) if code_indices else 0.0
    P_lang = np.sum(probs[lang_indices]) if lang_indices else 0.0
    P_other = np.sum(probs[other_indices]) if other_indices else 0.0
    
    H_code = entropy_from_probs(probs[code_indices])
    H_lang = entropy_from_probs(probs[lang_indices])
    H_total = entropy_from_probs(probs)
    
    CCE = (P_code * H_code) - (P_lang * H_lang)
    
    return {
        'uncertainty_type': int(uncertainty_type),
        'uncertainty_prob': float(uncertainty_prob),
        'p_code': float(P_code),
        'p_lang': float(P_lang),
        'p_other': float(P_other),
        'h_code': float(H_code),
        'h_lang': float(H_lang),
        'h_total': float(H_total),
        'cce': float(CCE),
    }

print("✅ Entropy computation functions ready")

## 3. Diverse Test Set (Validation of Real Learning)

In [ ]:
# Cell 9: Diverse test prompts

TEST_PROMPTS = [
    # ========== CODE UNCERTAINTY (expect uncertainty_type = 1, positive CCE) ==========
    
    # Different languages
    {'prompt': 'fn main() {', 'label': 1, 'category': 'rust_syntax', 'desc': 'Rust uncertain continuation'},
    {'prompt': 'package main\nimport "', 'label': 1, 'category': 'go_import', 'desc': 'Go uncertain import'},
    {'prompt': 'defmodule MyApp do\n  def', 'label': 1, 'category': 'elixir_function', 'desc': 'Elixir uncertain function'},
    {'prompt': 'class Main extends', 'label': 1, 'category': 'java_extends', 'desc': 'Java uncertain parent class'},
    {'prompt': 'use std::', 'label': 1, 'category': 'rust_use', 'desc': 'Rust uncertain module'},
    
    # Different domains
    {'prompt': 'import torch.nn.', 'label': 1, 'category': 'ml_import', 'desc': 'ML uncertain PyTorch module'},
    {'prompt': 'app.route(\'/api/', 'label': 1, 'category': 'web_route', 'desc': 'Web uncertain endpoint'},
    {'prompt': 'ssh user@', 'label': 1, 'category': 'sysadmin_ssh', 'desc': 'Sysadmin uncertain host'},
    {'prompt': 'kubectl apply -f', 'label': 1, 'category': 'devops_k8s', 'desc': 'DevOps uncertain file'},
    {'prompt': 'CREATE TABLE users (', 'label': 1, 'category': 'database_schema', 'desc': 'Database uncertain column'},
    
    # Niche/obscure libraries
    {'prompt': 'from Bio.', 'label': 1, 'category': 'niche_biopython', 'desc': 'Biopython uncertain module'},
    {'prompt': 'import plotly.graph_objects as go\nfig = go.', 'label': 1, 'category': 'niche_plotly', 'desc': 'Plotly uncertain chart'},
    {'prompt': 'connection = psycopg2.', 'label': 1, 'category': 'niche_postgres', 'desc': 'psycopg2 uncertain method'},
    {'prompt': 'stream = cv2.', 'label': 1, 'category': 'niche_opencv', 'desc': 'OpenCV uncertain function'},
    
    # Edge cases - incomplete statements
    {'prompt': 'if user.is_', 'label': 1, 'category': 'edge_incomplete', 'desc': 'Incomplete attribute'},
    {'prompt': 'result = requests.', 'label': 1, 'category': 'edge_method', 'desc': 'Uncertain HTTP method'},
    {'prompt': 'np.random.', 'label': 1, 'category': 'edge_numpy', 'desc': 'NumPy uncertain function'},
    
    # ========== LANGUAGE UNCERTAINTY (expect uncertainty_type = 0, negative CCE) ==========
    
    # Technical explanations
    {'prompt': 'The main advantage of microservices is', 'label': 0, 'category': 'tech_explain', 'desc': 'Technical explanation'},
    {'prompt': 'Machine learning models can', 'label': 0, 'category': 'ml_explain', 'desc': 'ML explanation'},
    {'prompt': 'Docker containers are useful because they', 'label': 0, 'category': 'devops_explain', 'desc': 'DevOps explanation'},
    
    # Comparisons
    {'prompt': 'The difference between REST and GraphQL is', 'label': 0, 'category': 'comparison', 'desc': 'API comparison'},
    {'prompt': 'Unlike synchronous code, asynchronous code', 'label': 0, 'category': 'comparison', 'desc': 'Async comparison'},
    
    # Instructions/questions
    {'prompt': 'Explain how binary search', 'label': 0, 'category': 'instruction', 'desc': 'Algorithm explanation request'},
    {'prompt': 'Describe the process of', 'label': 0, 'category': 'instruction', 'desc': 'Process description request'},
    {'prompt': 'What are the benefits of', 'label': 0, 'category': 'question', 'desc': 'Benefits question'},
    
    # Creative/metaphorical
    {'prompt': 'Write a poem about databases', 'label': 0, 'category': 'creative', 'desc': 'Poetry about tech'},
    {'prompt': 'If programming languages were animals, Python would be', 'label': 0, 'category': 'creative', 'desc': 'Metaphor'},
    {'prompt': 'A story about a bug that', 'label': 0, 'category': 'creative', 'desc': 'Story'},
    
    # Non-technical language
    {'prompt': 'The weather today is', 'label': 0, 'category': 'general', 'desc': 'Weather'},
    {'prompt': 'My favorite food is', 'label': 0, 'category': 'general', 'desc': 'Food preference'},
    {'prompt': 'Once upon a time, there was', 'label': 0, 'category': 'general', 'desc': 'Story beginning'},
    
    # ========== EDGE CASES / AMBIGUOUS ==========
    
    # Comments (language about code)
    {'prompt': '# This function calculates', 'label': 0, 'category': 'edge_comment', 'desc': 'Python comment'},
    {'prompt': '// Initialize the', 'label': 0, 'category': 'edge_comment', 'desc': 'JS comment'},
    
    # Docstrings
    {'prompt': '"""Returns the sum of', 'label': 0, 'category': 'edge_docstring', 'desc': 'Docstring'},
    
    # Mixed code/text in prompts
    {'prompt': 'Using the PySolarWinds library, connect to', 'label': 1, 'category': 'mixed', 'desc': 'Instruction with code uncertainty'},
    {'prompt': 'Explain what this code does: import', 'label': 0, 'category': 'mixed', 'desc': 'Explanation request'},
]

print(f"Test prompts: {len(TEST_PROMPTS)}")
print(f"  Code uncertainty (label=1): {sum(1 for p in TEST_PROMPTS if p['label'] == 1)}")
print(f"  Language uncertainty (label=0): {sum(1 for p in TEST_PROMPTS if p['label'] == 0)}")

# Show category distribution
categories = {}
for p in TEST_PROMPTS:
    cat = p['category']
    categories[cat] = categories.get(cat, 0) + 1

print(f"\nCategories: {len(categories)}")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

## 4. Run Experiments

In [ ]:
# Cell 10: Compute entropy for all test prompts

print("="*80)
print("COMPUTING ENTROPY FOR ALL TEST PROMPTS")
print("="*80)

results = []

for test_prompt in tqdm(TEST_PROMPTS, desc="Testing"):
    entropy_breakdown = compute_entropy_breakdown(
        test_prompt['prompt'],
        probe,
        scaler,
        SELECTED_LAYERS
    )
    
    result = {
        'prompt': test_prompt['prompt'],
        'true_label': test_prompt['label'],
        'category': test_prompt['category'],
        'desc': test_prompt['desc'],
        **entropy_breakdown
    }
    
    # Check if prediction matches
    result['correct'] = result['uncertainty_type'] == test_prompt['label']
    
    results.append(result)

df = pd.DataFrame(results)
print(f"\n✅ Computed entropy for {len(results)} prompts")

In [ ]:
# Cell 11: Display results

print("\n" + "="*100)
print("RESULTS")
print("="*100)

print(f"\n{'Category':<20} {'Prompt':<35} {'Type':<6} {'P(C_unc)':<10} {'CCE':<8} {'Status'}")
print("-"*100)

for _, row in df.iterrows():
    type_str = 'CODE' if row['uncertainty_type'] == 1 else 'LANG'
    status = '✅' if row['correct'] else '❌'
    prompt_short = row['prompt'][:35]
    
    print(f"{row['category']:<20} {prompt_short:<35} {type_str:<6} "
          f"{row['uncertainty_prob']:.3f}      {row['cce']:+.3f}   {status}")

## 5. Analysis

In [ ]:
# Cell 12: Statistical analysis

code_unc = df[df['true_label'] == 1]
lang_unc = df[df['true_label'] == 0]

# Accuracy
overall_accuracy = df['correct'].mean()
code_accuracy = code_unc['correct'].mean()
lang_accuracy = lang_unc['correct'].mean()

# CCE statistics
code_cce = code_unc['cce'].values
lang_cce = lang_unc['cce'].values

t_stat, p_value = ttest_ind(code_cce, lang_cce)
mean_diff = code_cce.mean() - lang_cce.mean()

print("\n" + "="*80)
print("STATISTICAL ANALYSIS")
print("="*80)

print(f"\n1. PROBE ACCURACY")
print(f"   Overall: {overall_accuracy:.1%} ({df['correct'].sum()}/{len(df)})")
print(f"   Code uncertainty: {code_accuracy:.1%}")
print(f"   Language uncertainty: {lang_accuracy:.1%}")

print(f"\n2. CCE SEPARATION")
print(f"   Code uncertainty (expect POSITIVE):")
print(f"     Mean CCE: {code_cce.mean():+.3f}")
print(f"     Std: {code_cce.std():.3f}")
print(f"     Range: [{code_cce.min():+.3f}, {code_cce.max():+.3f}]")

print(f"\n   Language uncertainty (expect NEGATIVE):")
print(f"     Mean CCE: {lang_cce.mean():+.3f}")
print(f"     Std: {lang_cce.std():.3f}")
print(f"     Range: [{lang_cce.min():+.3f}, {lang_cce.max():+.3f}]")

print(f"\n   Separation: {mean_diff:+.3f}")
print(f"   t-statistic: {t_stat:.3f}")
print(f"   p-value: {p_value:.6f}")
print(f"   Significant: {'YES ✅' if p_value < 0.05 else 'NO ❌'}")

print(f"\n3. ENTROPY BREAKDOWN")
print(f"   Code uncertainty:")
print(f"     H_code: {code_unc['h_code'].mean():.3f} ± {code_unc['h_code'].std():.3f}")
print(f"     H_lang: {code_unc['h_lang'].mean():.3f} ± {code_unc['h_lang'].std():.3f}")
print(f"     P_code: {code_unc['p_code'].mean():.3f} ± {code_unc['p_code'].std():.3f}")
print(f"     P_lang: {code_unc['p_lang'].mean():.3f} ± {code_unc['p_lang'].std():.3f}")

print(f"\n   Language uncertainty:")
print(f"     H_code: {lang_unc['h_code'].mean():.3f} ± {lang_unc['h_code'].std():.3f}")
print(f"     H_lang: {lang_unc['h_lang'].mean():.3f} ± {lang_unc['h_lang'].std():.3f}")
print(f"     P_code: {lang_unc['p_code'].mean():.3f} ± {lang_unc['p_code'].std():.3f}")
print(f"     P_lang: {lang_unc['p_lang'].mean():.3f} ± {lang_unc['p_lang'].std():.3f}")

# Save results
df.to_csv('week3_multi_layer_sep_results.csv', index=False)
print(f"\n✅ Results saved to: week3_multi_layer_sep_results.csv")

In [ ]:
# Cell 13: Category-wise analysis

print("\n" + "="*80)
print("CATEGORY-WISE PERFORMANCE")
print("="*80)

category_stats = df.groupby('category').agg({
    'correct': ['count', 'sum', 'mean'],
    'cce': ['mean', 'std'],
    'uncertainty_prob': 'mean'
}).round(3)

category_stats.columns = ['Count', 'Correct', 'Accuracy', 'Mean_CCE', 'Std_CCE', 'Mean_P(code_unc)']
category_stats = category_stats.sort_values('Accuracy', ascending=False)

print(category_stats.to_string())

# Identify challenging categories
challenging = category_stats[category_stats['Accuracy'] < 0.5]
if len(challenging) > 0:
    print(f"\n⚠️  Challenging categories (accuracy < 50%):")
    print(challenging.to_string())
else:
    print(f"\n✅ All categories have accuracy ≥ 50%")

In [ ]:
# Cell 14: Visualizations

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: CCE distribution
ax = axes[0, 0]
ax.hist(code_cce, bins=15, alpha=0.7, label='Code Uncertainty', color='coral')
ax.hist(lang_cce, bins=15, alpha=0.7, label='Language Uncertainty', color='steelblue')
ax.axvline(0, color='black', linestyle='--', linewidth=2)
ax.set_xlabel('CCE')
ax.set_ylabel('Count')
ax.set_title('CCE Distribution by Uncertainty Type')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Probability distribution
ax = axes[0, 1]
code_probs = code_unc['uncertainty_prob'].values
lang_probs = lang_unc['uncertainty_prob'].values
ax.hist(code_probs, bins=15, alpha=0.7, label='Code Uncertainty (true)', color='coral')
ax.hist(lang_probs, bins=15, alpha=0.7, label='Language Uncertainty (true)', color='steelblue')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision boundary')
ax.set_xlabel('P(Code Uncertainty)')
ax.set_ylabel('Count')
ax.set_title('Probe Probability Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: H_code vs H_lang
ax = axes[0, 2]
ax.scatter(code_unc['h_code'], code_unc['h_lang'], alpha=0.6, c='coral', label='Code Unc', s=50)
ax.scatter(lang_unc['h_code'], lang_unc['h_lang'], alpha=0.6, c='steelblue', label='Lang Unc', s=50)
ax.plot([0, 15], [0, 15], 'k--', alpha=0.3, label='H_code = H_lang')
ax.set_xlabel('H_code (bits)')
ax.set_ylabel('H_lang (bits)')
ax.set_title('Code Entropy vs Language Entropy')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: P_code vs P_lang
ax = axes[1, 0]
ax.scatter(code_unc['p_code'], code_unc['p_lang'], alpha=0.6, c='coral', label='Code Unc', s=50)
ax.scatter(lang_unc['p_code'], lang_unc['p_lang'], alpha=0.6, c='steelblue', label='Lang Unc', s=50)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='P_code = P_lang')
ax.set_xlabel('P_code')
ax.set_ylabel('P_lang')
ax.set_title('Code Probability Mass vs Language Probability Mass')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 5: Category accuracy
ax = axes[1, 1]
cat_acc = category_stats['Accuracy'].sort_values(ascending=True)
colors_cat = ['green' if x >= 0.7 else 'orange' if x >= 0.5 else 'red' for x in cat_acc.values]
cat_acc.plot(kind='barh', ax=ax, color=colors_cat)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Accuracy')
ax.set_title('Accuracy by Category')
ax.grid(True, alpha=0.3, axis='x')

# Plot 6: CCE vs Probability
ax = axes[1, 2]
ax.scatter(code_unc['uncertainty_prob'], code_unc['cce'], alpha=0.6, c='coral', label='Code Unc', s=50)
ax.scatter(lang_unc['uncertainty_prob'], lang_unc['cce'], alpha=0.6, c='steelblue', label='Lang Unc', s=50)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('P(Code Uncertainty)')
ax.set_ylabel('CCE')
ax.set_title('CCE vs Probe Probability')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week3_multi_layer_sep_analysis.png', dpi=150)
plt.show()

print("\n✅ Visualizations saved")

## 6. Final Summary

In [ ]:
# Cell 15: Summary

print("\n" + "="*80)
print("WEEK 3: MULTI-LAYER SEP - FINAL SUMMARY")
print("="*80)

print(f"\n1. METHOD")
print(f"   ✅ Multi-layer probe (layers {SELECTED_LAYERS})")
print(f"   ✅ SEP training objective (uncertainty TYPE not semantic mode)")
print(f"   ✅ Proper entropy computation (H_code, H_lang, CCE)")
print(f"   ✅ Diverse test set ({len(TEST_PROMPTS)} prompts, {len(categories)} categories)")

print(f"\n2. RESULTS")
print(f"   Training: {cv_accuracy:.0%} CV accuracy (LOO)")
print(f"   Testing: {overall_accuracy:.1%} accuracy on diverse prompts")
print(f"   CCE separation: {mean_diff:+.3f} (p={p_value:.4f})")

print(f"\n3. KEY FINDINGS")
if code_cce.mean() > 0:
    print(f"   ✅ Code uncertainty → POSITIVE CCE (mean: {code_cce.mean():+.3f})")
else:
    print(f"   ❌ Code uncertainty → negative CCE (mean: {code_cce.mean():+.3f})")

if lang_cce.mean() < 0:
    print(f"   ✅ Language uncertainty → NEGATIVE CCE (mean: {lang_cce.mean():+.3f})")
else:
    print(f"   ⚠️  Language uncertainty → positive CCE (mean: {lang_cce.mean():+.3f})")

if p_value < 0.05:
    print(f"   ✅ Statistically significant separation (p < 0.05)")
else:
    print(f"   ⚠️  Not statistically significant (p = {p_value:.4f})")

print(f"\n4. GENERALIZATION")
print(f"   Tested on unseen:")
print(f"     - Programming languages (Rust, Go, Elixir, Java)")
print(f"     - Domains (ML, web, sysadmin, DevOps, database)")
print(f"     - Edge cases (comments, docstrings, mixed)")
print(f"     - Creative prompts (poems, stories, metaphors)")
print(f"     - General language (weather, food, etc.)")

print(f"\n5. COMPARISON TO PREVIOUS METHODS")
print(f"   {'Method':<30} {'Train Acc':<12} {'Test Acc':<12} {'CCE Sep':<12}")
print(f"   {'-'*66}")
print(f"   {'Hardcoded PMR':<30} {'90%':<12} {'N/A':<12} {'✅ (p<0.05)':<12}")
print(f"   {'Final Integration':<30} {'100%':<12} {'0%':<12} {'❌ (p=0.57)':<12}")
print(f"   {'Single-Layer SEP':<30} {'100%':<12} {'94% (OOD)':<12} {'Unknown':<12}")
print(f"   {'Multi-Layer SEP (THIS)':<30} {f'{cv_accuracy:.0%}':<12} {f'{overall_accuracy:.1%}':<12} {f'p={p_value:.3f}':<12}")

print(f"\n6. RECOMMENDATION")
if overall_accuracy >= 0.7 and (p_value < 0.05 or mean_diff > 2.0):
    print(f"   ✅ USE THIS METHOD for Week 3")
    print(f"   Reasons:")
    print(f"     - High accuracy on diverse test set")
    print(f"     - Proper training objective (uncertainty type)")
    print(f"     - Multi-layer representation")
    print(f"     - Good CCE separation")
elif overall_accuracy >= 0.7:
    print(f"   ⚠️  PROMISING but needs refinement")
    print(f"   - Good accuracy but CCE separation not significant")
    print(f"   - Consider: spike detection, different layers, more training data")
else:
    print(f"   ❌ NEEDS MORE WORK")
    print(f"   - Accuracy too low on diverse test set")
    print(f"   - Probe may be overfitting to training examples")

print(f"\n" + "="*80)